# Urdu + English voice reminder: train the speech model

This trains a small speech model **from scratch** (random start, no Whisper or other pretrained model).
It hears Urdu (including what people write as Roman Urdu) and English, and outputs phonetic sounds,
so tablet names and times can be matched afterwards.

**Before you start:** *Runtime → Change runtime type → T4 GPU*, then *Runtime → Run all*.

- Data: Common Voice Urdu (CC0), LibriSpeech English, and spoken medicine commands.
- Checkpoints are saved to your Google Drive after every epoch, in `MyDrive/voice_reminder/checkpoints`.
- **If Colab disconnects**, just run all cells again. The data steps repeat (about 20 to 30 minutes),
  then training continues from the last saved epoch.
- Optional: put `commands.zip` (made on the Mac) in `MyDrive/voice_reminder/`. Those clips use more
  natural voices than the robotic ones this notebook generates by itself.

In [ ]:
#@title 1. Check the GPU
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo "NO GPU: choose Runtime > Change runtime type > T4 GPU"

In [ ]:
#@title 2. Settings and Google Drive
EPOCHS = 30      #@param {type:"integer"}
DIM = 144        #@param {type:"integer"}
BLOCKS = 5       #@param {type:"integer"}
RUN_NAME = "colab_model"  #@param {type:"string"}

from google.colab import drive
drive.mount('/content/drive')
import os
DRIVE_DIR = '/content/drive/MyDrive/voice_reminder'
CKPT_DIR = DRIVE_DIR + '/checkpoints'
os.makedirs(CKPT_DIR, exist_ok=True)
os.makedirs('/content/hijab/src', exist_ok=True)
os.makedirs('/content/hijab/logs', exist_ok=True)
%cd /content/hijab

In [ ]:
#@title 3. Install tools
!apt-get -qq install -y espeak-ng > /dev/null && espeak-ng --version
!pip -q install librosa soundfile pyarrow

## 4. Project code
These cells write the same source files used on the Mac.

In [ ]:
%%writefile /content/hijab/src/phones.py
"""Text to phoneme tokens, shared by Urdu and English.

Both languages go through espeak-ng's rule-based converter and are then folded into
one small phone set. The model therefore has a single output alphabet, so mixed
speech such as "Panadol subah aath baje" has one consistent transcription, and a new
tablet name can be recognised just by converting its spelling to phones.
"""
import re
import subprocess
import unicodedata
from concurrent.futures import ThreadPoolExecutor

BLANK = "<b>"
SPACE = "|"

# Fold espeak's detailed IPA into coarser classes. Distinctions that vary with accent
# (vowel length, retroflex r, the several schwa-like vowels) carry little information
# for picking a tablet name and a time, and merging them makes training easier.
FOLD = {
    "ɐ": "ə", "ʌ": "ə", "ɜ": "ə", "ᵻ": "ɪ", "ɨ": "ɪ", "y": "i",
    "ɑ": "a", "æ": "a", "ɒ": "ɔ", "o": "ɔ", "e": "ɛ",
    "ɾ": "r", "ɹ": "r", "ɽ": "r", "ɻ": "r", "ʁ": "r", "ɚ": "ər", "ɝ": "ər",
    "ɡ": "g", "ç": "h", "x": "h", "ɦ": "h", "ʔ": "",
    "ʋ": "v", "ɬ": "l", "ɫ": "l", "ɭ": "l",
    "ɳ": "n", "ɲ": "n", "ŋ": "n",
    "ʂ": "s", "ʒ": "z", "ɣ": "g", "χ": "h", "ħ": "h", "ʕ": "", "q": "k",
    "ʈ": "t", "ɖ": "d", "ð": "d", "θ": "t",
    "c": "ʧ", "ɟ": "ʤ",
    "ʰ": "h", "ʱ": "h", "ʐ": "z",
    # Nasal vowels compose into single characters under NFC, so they must be folded here.
    "ẽ": "ɛ", "õ": "ɔ", "ã": "a", "ũ": "u", "ĩ": "i",
}
MARKS = re.compile(r"\([a-z-]+\)|[ˈˌːˑ‿̩̪̥̬̯̃͡ʲˠˤ\-_.,;:!?'\"]")

VOCAB = [BLANK, SPACE] + list("abdfghijklmnprstuvwzɔəɛɪʃʊʧʤ")
TOKEN_ID = {t: i for i, t in enumerate(VOCAB)}


def espeak(text: str, voice: str) -> str:
    # Text goes in as an argument: with --stdin, espeak-ng reads past the end of
    # non-ASCII input and appends spurious syllables.
    text = " ".join(text.split()).lstrip("-") or " "
    try:
        out = subprocess.run(["espeak-ng", "-q", "--ipa", "-v", voice, text], capture_output=True, timeout=30)
    except subprocess.TimeoutExpired:
        return ""  # an empty result is dropped by the caller
    return " ".join(out.stdout.decode("utf-8", "ignore").split())


def fold(ipa: str, unknown: dict | None = None) -> list[str]:
    ipa = MARKS.sub("", unicodedata.normalize("NFC", ipa))
    ipa = ipa.replace("dʒ", "ʤ").replace("tʃ", "ʧ")
    tokens: list[str] = []
    for ch in ipa:
        if ch == " ":
            if tokens and tokens[-1] != SPACE:
                tokens.append(SPACE)
            continue
        for mapped in FOLD.get(ch, ch):
            if mapped in TOKEN_ID:
                tokens.append(mapped)
            elif unknown is not None:
                unknown[mapped] = unknown.get(mapped, 0) + 1
    while tokens and tokens[-1] == SPACE:
        tokens.pop()
    return tokens


def to_tokens(text: str, voice: str, unknown: dict | None = None) -> list[str]:
    return fold(espeak(text, voice), unknown)


def to_tokens_many(items: list[tuple[str, str]], workers: int = 10, unknown: dict | None = None) -> list[list[str]]:
    """Convert (text, voice) pairs, calling espeak once per distinct pair."""
    distinct = sorted(set(items))
    with ThreadPoolExecutor(workers) as pool:
        ipa = dict(zip(distinct, pool.map(lambda p: espeak(p[0], p[1]), distinct)))
    return [fold(ipa[i], unknown) for i in items]

In [ ]:
%%writefile /content/hijab/src/features.py
"""Log-mel features: 16 kHz audio, 25 ms window, 10 ms hop, 80 bands."""
import numpy as np
import librosa

SR = 16000
N_FFT = 400
HOP = 160
N_MELS = 80
_WINDOW = np.hanning(N_FFT).astype(np.float32)
_MEL = librosa.filters.mel(sr=SR, n_fft=N_FFT, n_mels=N_MELS, fmin=40, fmax=7600).astype(np.float32)


def log_mel(wav: np.ndarray) -> np.ndarray:
    """Return an array of shape (frames, 80) from mono float audio at 16 kHz."""
    if len(wav) < N_FFT:
        wav = np.pad(wav, (0, N_FFT - len(wav)))
    n = 1 + (len(wav) - N_FFT) // HOP
    idx = np.arange(N_FFT)[None] + HOP * np.arange(n)[:, None]
    spec = np.abs(np.fft.rfft(wav[idx] * _WINDOW, axis=1)) ** 2
    return np.log(np.maximum(spec @ _MEL.T, 1e-10)).astype(np.float32)


def normalize(mel: np.ndarray) -> np.ndarray:
    """Zero mean and unit variance per utterance, so loudness and mic gain drop out."""
    return (mel - mel.mean(0, keepdims=True)) / (mel.std(0, keepdims=True) + 1e-5)

In [ ]:
%%writefile /content/hijab/src/model.py
"""A compact Conformer acoustic model, initialized randomly (no pretrained weights)."""
import math

import torch
import torch.nn as nn
import torch.nn.functional as F


class ConvSubsample(nn.Module):
    """Reduce time by 4 and project the mel bands to the model dimension."""

    def __init__(self, n_mels: int, dim: int, channels: int = 32):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(1, channels, 3, stride=2, padding=1), nn.ReLU(),
            nn.Conv2d(channels, channels, 3, stride=2, padding=1), nn.ReLU(),
        )
        self.proj = nn.Linear(channels * math.ceil(math.ceil(n_mels / 2) / 2), dim)

    def forward(self, x: torch.Tensor, lengths: torch.Tensor):
        x = self.conv(x.unsqueeze(1)).permute(0, 2, 1, 3).flatten(2)
        return self.proj(x), ((lengths + 3) // 4).clamp(max=x.size(1))


class FeedForward(nn.Sequential):
    def __init__(self, dim: int, dropout: float):
        super().__init__(nn.LayerNorm(dim), nn.Linear(dim, dim * 4), nn.SiLU(), nn.Dropout(dropout),
                         nn.Linear(dim * 4, dim), nn.Dropout(dropout))


class ConvModule(nn.Module):
    def __init__(self, dim: int, kernel: int, dropout: float):
        super().__init__()
        self.norm = nn.LayerNorm(dim)
        self.net = nn.Sequential(
            nn.Conv1d(dim, dim * 2, 1), nn.GLU(dim=1),
            nn.Conv1d(dim, dim, kernel, padding=kernel // 2, groups=dim),
            nn.GroupNorm(1, dim), nn.SiLU(), nn.Conv1d(dim, dim, 1), nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(self.norm(x).transpose(1, 2)).transpose(1, 2)


class ConformerBlock(nn.Module):
    def __init__(self, dim: int, heads: int, kernel: int, dropout: float):
        super().__init__()
        self.ff1 = FeedForward(dim, dropout)
        self.attn_norm = nn.LayerNorm(dim)
        self.attn = nn.MultiheadAttention(dim, heads, dropout=dropout, batch_first=True)
        self.drop = nn.Dropout(dropout)
        self.conv = ConvModule(dim, kernel, dropout)
        self.ff2 = FeedForward(dim, dropout)
        self.norm = nn.LayerNorm(dim)

    def forward(self, x, pad_mask):
        x = x + 0.5 * self.ff1(x)
        y = self.attn_norm(x)
        x = x + self.drop(self.attn(y, y, y, key_padding_mask=pad_mask, need_weights=False)[0])
        x = x + self.conv(x)
        x = x + 0.5 * self.ff2(x)
        return self.norm(x)


class SpeechModel(nn.Module):
    def __init__(self, n_tokens: int, n_mels: int = 80, dim: int = 144, blocks: int = 5, heads: int = 4,
                 kernel: int = 15, dropout: float = 0.1):
        super().__init__()
        self.config = dict(n_tokens=n_tokens, n_mels=n_mels, dim=dim, blocks=blocks, heads=heads, kernel=kernel, dropout=dropout)
        self.sub = ConvSubsample(n_mels, dim)
        self.pos = nn.Parameter(torch.zeros(1, 4000, dim))
        nn.init.trunc_normal_(self.pos, std=0.02)
        self.drop = nn.Dropout(dropout)
        self.blocks = nn.ModuleList(ConformerBlock(dim, heads, kernel, dropout) for _ in range(blocks))
        self.out = nn.Linear(dim, n_tokens)

    def forward(self, x: torch.Tensor, lengths: torch.Tensor):
        x, lengths = self.sub(x, lengths)
        x = self.drop(x + self.pos[:, : x.size(1)])
        mask = torch.arange(x.size(1), device=x.device)[None] >= lengths[:, None]
        for block in self.blocks:
            x = block(x, mask)
        return F.log_softmax(self.out(x), -1), lengths

In [ ]:
%%writefile /content/hijab/src/download_data.py
"""Download raw speech data into data/raw/.

Urdu:    Common Voice Urdu parquet shards (public, no login).
English: a capped portion of LibriSpeech train-clean-100, streamed so that only
         the audio we keep ever reaches the disk.

Downloaded files are treated as data only: parquet is read as columns, and tar
members are written to paths we build ourselves, never to paths from the archive.
"""
import re
import sys
import tarfile
import time
import urllib.request
from pathlib import Path

ROOT = Path(__file__).resolve().parent.parent
RAW = ROOT / "data" / "raw"
UA = {"User-Agent": "voice-reminder-trainer/1.0"}

URDU_REPO = "khawajaaliarshad/common-voice-urdu-processed-expanded"
URDU_FILES = [
    "README.md",
    "data/train-00000-of-00003.parquet",
    "data/train-00001-of-00003.parquet",
    "data/train-00002-of-00003.parquet",
    "data/validation-00000-of-00001.parquet",
    "data/test-00000-of-00001.parquet",
]
LIBRI_URL = "https://www.openslr.org/resources/12/train-clean-100.tar.gz"
LIBRI_MAX_BYTES = int(2.0e9)  # about 25 hours of FLAC


def fetch(url: str, dest: Path) -> None:
    if dest.exists() and dest.stat().st_size > 0:
        print(f"have {dest.name}", flush=True)
        return
    dest.parent.mkdir(parents=True, exist_ok=True)
    tmp = dest.with_suffix(dest.suffix + ".part")
    with urllib.request.urlopen(urllib.request.Request(url, headers=UA), timeout=120) as r, open(tmp, "wb") as f:
        while chunk := r.read(1 << 20):
            f.write(chunk)
    tmp.rename(dest)
    print(f"got {dest.name} {dest.stat().st_size/1e6:.0f} MB", flush=True)


def download_urdu() -> None:
    base = f"https://huggingface.co/datasets/{URDU_REPO}/resolve/main/"
    for name in URDU_FILES:
        fetch(base + name, RAW / "urdu_cv" / Path(name).name)


def download_english() -> None:
    out = RAW / "librispeech"
    done = out / ".complete"
    if done.exists():
        print("have librispeech", flush=True)
        return
    out.mkdir(parents=True, exist_ok=True)
    flac = re.compile(r"LibriSpeech/train-clean-100/(\d+)/(\d+)/(\d+-\d+-\d+)\.flac$")
    trans = re.compile(r"LibriSpeech/train-clean-100/(\d+)/(\d+)/(\d+-\d+)\.trans\.txt$")
    written = clips = 0
    t0 = time.time()
    with urllib.request.urlopen(urllib.request.Request(LIBRI_URL, headers=UA), timeout=120) as r:
        with tarfile.open(fileobj=r, mode="r|gz") as tar:
            for m in tar:
                if not m.isfile():
                    continue
                if mf := flac.match(m.name):
                    if written >= LIBRI_MAX_BYTES:
                        break
                    target = out / f"{mf.group(3)}.flac"
                elif mt := trans.match(m.name):
                    target = out / f"{mt.group(3)}.trans.txt"
                else:
                    continue
                data = tar.extractfile(m).read()
                target.write_bytes(data)
                if target.suffix == ".flac":
                    written += len(data)
                    clips += 1
                    if clips % 2000 == 0:
                        print(f"librispeech {clips} clips {written/1e9:.2f} GB {time.time()-t0:.0f}s", flush=True)
    done.write_text(f"{clips} clips\n")
    print(f"librispeech done: {clips} clips, {written/1e9:.2f} GB", flush=True)


if __name__ == "__main__":
    which = sys.argv[1] if len(sys.argv) > 1 else "all"
    if which in ("all", "urdu"):
        download_urdu()
    if which in ("all", "english"):
        download_english()

In [ ]:
%%writefile /content/hijab/src/generate_commands.py
"""Synthesize spoken reminder commands so the model hears tablet names and clock times.

Public speech corpora almost never contain "Panadol subah aath baje". These clips use
the system voices purely to make training audio. Six tablet names and one English
voice are withheld for testing, so the test set contains things never trained on.

Every clip records its intended tablet and time for task-level evaluation later.
"""
import json
import random
import shutil
import subprocess
import tempfile
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

ROOT = Path(__file__).resolve().parent.parent
OUT = ROOT / "data" / "raw" / "commands"

# (Latin, Urdu script, Devanagari)
DRUGS = [
    ("Panadol", "پیناڈول", "पैनाडोल"), ("Augmentin", "آگمینٹن", "ऑगमेंटिन"), ("Brufen", "بروفین", "ब्रूफेन"),
    ("Disprin", "ڈسپرین", "डिस्प्रिन"), ("Flagyl", "فلیجل", "फ्लैजिल"), ("Risek", "رائزیک", "राइज़ेक"),
    ("Omeprazole", "اومیپرازول", "ओमेप्राज़ोल"), ("Metformin", "میٹفارمن", "मेटफॉर्मिन"), ("Amoxil", "اموکسل", "अमोक्सिल"),
    ("Ponstan", "پونسٹان", "पोंस्टान"), ("Calpol", "کیلپول", "कैलपोल"), ("Aspirin", "ایسپرین", "एस्पिरिन"),
    ("Insulin", "انسولین", "इंसुलिन"), ("Glucophage", "گلوکوفیج", "ग्लूकोफेज"), ("Concor", "کونکور", "कॉनकोर"),
    ("Lipitor", "لپیٹر", "लिपिटर"), ("Zyrtec", "زرٹیک", "ज़िरटेक"), ("Softin", "سافٹن", "सॉफ्टिन"),
    ("Arinac", "ارینیک", "एरिनैक"), ("Neurobion", "نیوروبیون", "न्यूरोबियॉन"), ("Surbex", "سربیکس", "सरबेक्स"),
    ("Caltrate", "کیلٹریٹ", "कैल्ट्रेट"), ("Nexium", "نیکسیم", "नेक्सियम"), ("Cipro", "سپرو", "सिप्रो"),
    ("Velosef", "ویلوسیف", "वेलोसेफ"), ("Klaricid", "کلیرسڈ", "क्लैरिसिड"), ("Losartan", "لوسارٹن", "लोसार्टन"),
    ("Amlodipine", "ایملوڈپین", "एम्लोडिपिन"), ("Atorvastatin", "اٹورواسٹیٹن", "एटोरवास्टेटिन"),
    ("Gaviscon", "گیوسکان", "गैविस्कॉन"), ("Tramal", "ٹرامل", "ट्रामल"), ("Xanax", "زینیکس", "ज़ैनेक्स"),
    ("Lexotanil", "لیکسوٹینل", "लेक्सोटैनिल"), ("Cetirizine", "سیٹیریزین", "सेटिरिज़ीन"),
    ("Paracetamol", "پیراسیٹامول", "पैरासिटामोल"), ("Vitamin D", "وٹامن ڈی", "विटामिन डी"),
]
HELD_OUT_DRUGS = [
    ("Ventolin", "وینٹولن", "वेंटोलिन"), ("Thyroxine", "تھائیروکسن", "थायरोक्सिन"), ("Mucaine", "میوکین", "म्यूकेन"),
    ("Nuberol", "نیوبرول", "न्यूबेरोल"), ("Motilium", "موٹیلیم", "मोटिलियम"), ("Entamizole", "اینٹامیزول", "एंटामिज़ोल"),
]

UR_NUM = "ایک دو تین چار پانچ چھ سات آٹھ نو دس گیارہ بارہ".split()
HI_NUM = "एक दो तीन चार पाँच छह सात आठ नौ दस ग्यारह बारह".split()
EN_NUM = "one two three four five six seven eight nine ten eleven twelve".split()
# (name, Urdu, Hindi, English, is_pm_for_hours)
PERIODS = [
    ("morning", "صبح", "सुबह", "am"), ("afternoon", "دوپہر", "दोपहर", "pm"),
    ("evening", "شام", "शाम", "pm"), ("night", "رات", "रात", "pm"),
]
UR_MIN = {5: "پانچ", 10: "دس", 15: "پندرہ", 20: "بیس", 25: "پچیس", 40: "چالیس", 50: "پچاس"}
HI_MIN = {5: "पाँच", 10: "दस", 15: "पंद्रह", 20: "बीस", 25: "पच्चीस", 40: "चालीस", 50: "पचास"}
EN_MIN = {5: "oh five", 10: "ten", 15: "fifteen", 20: "twenty", 25: "twenty five", 30: "thirty", 40: "forty", 45: "forty five", 50: "fifty"}

UR_T = ["{d} {p} {t} کھانی ہے", "مجھے {p} {t} {d} یاد دلانا", "{p} {t} {d} کی گولی", "{d} روزانہ {p} {t}",
        "{p} {t} مجھے {d} لینی ہے", "{d} کا الارم {p} {t} لگا دو"]
HI_T = ["{d} {p} {t} खानी है", "मुझे {p} {t} {d} याद दिलाना", "{p} {t} {d} की गोली", "{d} रोज़ाना {p} {t}",
        "{p} {t} मुझे {d} लेनी है", "{d} का अलार्म {p} {t} लगा दो"]
EN_T = ["take {d} at {t}", "remind me to take {d} at {t}", "{d} at {t} every day", "set an alarm for {d} at {t}",
        "i need my {d} tablet at {t}", "{d} tablet at {t}"]

EN_VOICES = ["Daniel", "Samantha", "Moira", "Karen", "Tessa", "Fred", "Ralph", "Kathy", "Junior",
             "Eddy (English (UK))", "Flo (English (US))", "Reed (English (UK))", "Sandy (English (US))", "Shelley (English (UK))"]
EN_TEST_VOICE = "Rishi"  # the South Asian English voice, withheld from training
HINDI_VOICE = ("say", "Lekha")
EN_ENGINE = "say"

if shutil.which("say") is None:
    # Not on a Mac (for example on Colab): use espeak-ng voices for everything.
    EN_VOICES = ["en-us", "en-gb", "en-gb-scotland", "en-gb-x-rp", "en-gb-x-gbcwmd", "en-029",
                 "en-us+f3", "en-us+m3", "en-gb+f4", "en-us+f2", "en-gb+m4"]
    EN_TEST_VOICE = "en-gb-x-gbclan"
    HINDI_VOICE = ("espeak", "hi")
    EN_ENGINE = "espeak"


def ur_hi_time(rng: random.Random, h: int) -> tuple[str, str, int]:
    """Return an Urdu phrase, a matching Hindi phrase, and the minute."""
    style = rng.choice(["plain", "plain", "half", "quarter_past", "quarter_to", "minutes"])
    if style == "half":
        if h == 1:
            return "ڈیڑھ بجے", "डेढ़ बजे", 30
        if h == 2:
            return "ڈھائی بجے", "ढाई बजे", 30
        return f"ساڑھے {UR_NUM[h-1]} بجے", f"साढ़े {HI_NUM[h-1]} बजे", 30
    if style == "quarter_past":
        return f"سوا {UR_NUM[h-1]} بجے", f"सवा {HI_NUM[h-1]} बजे", 15
    if style == "quarter_to":
        return f"پونے {UR_NUM[h-1]} بجے", f"पौने {HI_NUM[h-1]} बजे", 45
    if style == "minutes":
        m = rng.choice(list(UR_MIN))
        return f"{UR_NUM[h-1]} بج کر {UR_MIN[m]} منٹ", f"{HI_NUM[h-1]} बज कर {HI_MIN[m]} मिनट", m
    return f"{UR_NUM[h-1]} بجے", f"{HI_NUM[h-1]} बजे", 0


def make_items(rng: random.Random, n_each: int) -> list[dict]:
    items = []
    for held, drugs, count in [(False, DRUGS, n_each), (True, HELD_OUT_DRUGS, n_each // 8)]:
        for _ in range(count):
            d = rng.choice(drugs)
            hour = rng.randint(1, 12)
            period = rng.choice(PERIODS)
            ti = rng.randrange(len(UR_T))
            tu, th, minute = ur_hi_time(rng, hour)
            # Quarter-to names the coming hour, so the clock hour is one less.
            clock = hour - 1 or 12 if minute == 45 else hour
            ur = UR_T[ti].format(d=d[1], p=period[1], t=tu)
            hi = HI_T[ti].format(d=d[2], p=period[2], t=th)
            base = {"drug": d[0], "hour": clock, "minute": minute, "period": period[0], "held_out_drug": held}
            items.append({**base, "lang": "ur", "engine": "espeak", "voice": "ur", "say": ur, "label": ur, "label_voice": "ur"})
            items.append({**base, "lang": "ur", "engine": HINDI_VOICE[0], "voice": HINDI_VOICE[1], "say": hi, "label": ur, "label_voice": "ur"})

            en_hour = rng.randint(1, 12)
            en_min = rng.choice([0, 0, 0, *EN_MIN])
            ampm = rng.choice(["am", "pm"])
            en_time = f"{EN_NUM[en_hour-1]} {EN_MIN[en_min]} {ampm}" if en_min else f"{EN_NUM[en_hour-1]} {ampm}"
            en = rng.choice(EN_T).format(d=d[0].lower(), t=en_time)
            en_base = {"drug": d[0], "hour": en_hour, "minute": en_min, "period": ampm, "held_out_drug": held}
            voice = EN_TEST_VOICE if rng.random() < 0.08 else rng.choice(EN_VOICES)
            items.append({**en_base, "lang": "en", "engine": EN_ENGINE, "voice": voice, "say": en, "label": en, "label_voice": "en-us"})
    return items


def render(job: tuple[int, dict, random.Random]) -> dict | None:
    i, it, rng = job
    wav = OUT / f"cmd_{i:06d}.wav"
    if it["engine"] == "espeak":
        cmd = ["espeak-ng", "-v", it["voice"], "-s", str(rng.randint(120, 175)), "-p", str(rng.randint(30, 70)), "-w", str(wav), it["say"]]
    else:
        cmd = ["say", "-v", it["voice"], "-r", str(rng.randint(140, 205)), "--data-format=LEI16@16000", "-o", str(wav), it["say"]]
    r = subprocess.run(cmd, capture_output=True, timeout=120)
    if r.returncode or not wav.exists() or wav.stat().st_size < 2000:
        return None
    return {**it, "wav": wav.name}


def main() -> None:
    OUT.mkdir(parents=True, exist_ok=True)
    rng = random.Random(1234)
    items = make_items(rng, 2200)
    jobs = [(i, it, random.Random(i)) for i, it in enumerate(items)]
    with ThreadPoolExecutor(8) as pool:
        done = [r for r in pool.map(render, jobs) if r]
    with open(OUT / "manifest.jsonl", "w", encoding="utf-8") as f:
        for r in done:
            r["split"] = "test" if r["held_out_drug"] or r["voice"] == EN_TEST_VOICE else "train"
            f.write(json.dumps(r, ensure_ascii=False) + "\n")
    print(f"commands: {len(done)}/{len(items)} rendered")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /content/hijab/src/prepare.py
"""Turn raw audio into one feature file plus a manifest, split by speaker.

Output:
  data/features/mels.bin        float16 log-mels, all clips concatenated
  data/features/manifest.jsonl  one row per clip (offset, frames, tokens, split, ...)

No speaker appears in more than one split, so validation and test scores measure how
the model does on voices it has never heard.
"""
import hashlib
import io
import json
import os
import re
import sys
from collections import Counter
from multiprocessing import Pool
from pathlib import Path

os.environ.setdefault("OMP_NUM_THREADS", "1")
sys.path.insert(0, str(Path(__file__).parent))

import librosa
import numpy as np
import pyarrow.parquet as pq
import soundfile as sf

import features as fx
import phones

ROOT = Path(__file__).resolve().parent.parent
RAW = ROOT / "data" / "raw"
OUT = ROOT / "data" / "features"
MIN_SEC, MAX_SEC = 0.4, 16.0
MAX_PHONES_PER_SEC = 22  # faster than people speak, so the transcript cannot match the audio
CORRUPT_TEXT = re.compile(r"[0-9a-f]{32}|twenties|thirties|fourties|fifties|teens|sixties|_feminine|_masculine|\t")


def speaker_split(speaker: str) -> str:
    bucket = int(hashlib.sha1(speaker.encode()).hexdigest(), 16) % 100
    return "test" if bucket < 6 else "val" if bucket < 12 else "train"


def urdu_rows():
    seen = set()
    stats = Counter()
    for path in sorted((RAW / "urdu_cv").glob("*.parquet")):
        table = pq.ParquetFile(path)
        for batch in table.iter_batches(batch_size=256, columns=["client_id", "sentence", "audio", "up_votes", "down_votes"]):
            for row in batch.to_pylist():
                stats["urdu_rows"] += 1
                key = (row["client_id"], row["sentence"])
                if key in seen:
                    stats["urdu_duplicates"] += 1
                    continue
                seen.add(key)
                # Some rows in the upload have neighbouring TSV rows merged into the
                # transcript, leaving speaker hashes and metadata in the text.
                if CORRUPT_TEXT.search(row["sentence"]) or len(row["sentence"]) > 300:
                    stats["urdu_corrupt_text"] += 1
                    continue
                if row["down_votes"] > row["up_votes"]:
                    stats["urdu_downvoted"] += 1
                    continue
                yield {"bytes": row["audio"]["bytes"], "text": row["sentence"], "voice": "ur", "speaker": "ur" + row["client_id"][:16],
                       "lang": "ur", "source": "common_voice"}
    print(dict(stats), flush=True)


def english_rows():
    base = RAW / "librispeech"
    for trans in sorted(base.glob("*.trans.txt")):
        for line in trans.read_text().splitlines():
            uid, text = line.split(" ", 1)
            wav = base / f"{uid}.flac"
            if wav.exists():
                yield {"file": str(wav), "text": text.lower(), "voice": "en-us", "speaker": "en" + uid.split("-")[0],
                       "lang": "en", "source": "librispeech"}


def command_rows():
    base = RAW / "commands"
    if not (base / "manifest.jsonl").exists():
        print("no command clips found, skipping them", flush=True)
        return
    for line in open(base / "manifest.jsonl", encoding="utf-8"):
        r = json.loads(line)
        yield {"file": str(base / r["wav"]), "text": r["label"], "voice": r["label_voice"], "speaker": "syn" + r["voice"], "lang": r["lang"],
               "source": "command", "split": r["split"], "drug": r["drug"], "hour": r["hour"], "minute": r["minute"], "period": r["period"],
               "held_out_drug": r["held_out_drug"], "say_voice": r["voice"]}


def work(row: dict):
    try:
        src = io.BytesIO(row.pop("bytes")) if "bytes" in row else row["file"]
        wav, sr = sf.read(src, dtype="float32", always_2d=True)
    except Exception as e:
        return None, f"decode:{type(e).__name__}"
    wav = wav.mean(1)
    if sr != fx.SR:
        wav = librosa.resample(wav, orig_sr=sr, target_sr=fx.SR)
    sec = len(wav) / fx.SR
    if not MIN_SEC <= sec <= MAX_SEC:
        return None, "duration"
    # Trim silence at the ends, keeping a short margin.
    intervals = librosa.effects.split(wav, top_db=40, frame_length=400, hop_length=160)
    if len(intervals):
        wav = wav[max(0, intervals[0][0] - 1600): intervals[-1][1] + 1600]
    mel = fx.log_mel(wav).astype(np.float16)
    row.pop("file", None)
    row["frames"] = len(mel)
    return (row, mel.tobytes()), None


def main() -> None:
    OUT.mkdir(parents=True, exist_ok=True)
    rows = []
    def sources():
        yield from urdu_rows()
        yield from english_rows()
        yield from command_rows()

    offset = 0
    drops = Counter()
    with Pool(os.cpu_count() or 1) as pool, open(OUT / "mels.bin", "wb") as bin_f:
        for i, (ok, why) in enumerate(pool.imap(work, sources(), chunksize=32)):
            if ok is None:
                drops[why] += 1
                continue
            row, data = ok
            row["offset"] = offset
            offset += row["frames"]
            bin_f.write(data)
            rows.append(row)
            if len(rows) % 5000 == 0:
                print(f"{len(rows)} clips, {offset/360000:.1f} h", flush=True)

    print("converting text to phones", flush=True)
    unknown: dict = {}
    tokens = phones.to_tokens_many([(r["text"], r["voice"]) for r in rows], unknown=unknown)
    kept, splits = [], Counter()
    for r, t in zip(rows, tokens):
        # CTC needs at least one output frame per label after 4x subsampling.
        if not t or len(t) > r["frames"] // 4 - 1:
            drops["label_length"] += 1
            continue
        if len(t) > r["frames"] / 100 * MAX_PHONES_PER_SEC:
            drops["label_rate"] += 1
            continue
        r["tokens"] = " ".join(t)
        r["split"] = r.get("split") or speaker_split(r["speaker"])
        splits[(r["source"], r["split"])] += r["frames"]
        kept.append(r)
    with open(OUT / "manifest.jsonl", "w", encoding="utf-8") as f:
        for r in kept:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")
    print("kept", len(kept), "dropped", dict(drops))
    print("hours by source and split:")
    for k, v in sorted(splits.items()):
        print("  ", k, round(v / 360000, 2))
    print("unknown phones (top):", Counter(unknown).most_common(12))


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /content/hijab/src/train.py
"""Train the speech model. Runs on an NVIDIA GPU (Colab), the Apple GPU, or the CPU.

Safe to stop and resume with --resume: a checkpoint is written after every epoch.
"""
import argparse
import json
import os
import random
import sys
import time
from pathlib import Path

sys.path.insert(0, str(Path(__file__).parent))

import numpy as np
import torch
import torch.nn.functional as F

import features as fx
import phones
from model import SpeechModel

ROOT = Path(__file__).resolve().parent.parent
FEAT = ROOT / "data" / "features"


def edit_distance(a, b) -> int:
    prev = list(range(len(b) + 1))
    for i, x in enumerate(a, 1):
        cur = [i]
        for j, y in enumerate(b, 1):
            cur.append(min(prev[j] + 1, cur[-1] + 1, prev[j - 1] + (x != y)))
        prev = cur
    return prev[-1]


class Data:
    def __init__(self):
        self.rows = [json.loads(l) for l in open(FEAT / "manifest.jsonl", encoding="utf-8")]
        for r in self.rows:
            r["ids"] = [phones.TOKEN_ID[t] for t in r["tokens"].split()]
        self._mels = None

    @property
    def mels(self):
        # Opened lazily so each data-loader worker maps the file itself rather than
        # receiving a pickled copy of several gigabytes.
        if self._mels is None:
            self._mels = np.memmap(FEAT / "mels.bin", dtype=np.float16, mode="r").reshape(-1, fx.N_MELS)
        return self._mels

    def __getstate__(self):
        return {**self.__dict__, "_mels": None}

    def split(self, name, source=None):
        return [r for r in self.rows if r["split"] == name and (source is None or r["source"] == source)]

    def load(self, r, train: bool) -> np.ndarray:
        m = np.array(self.mels[r["offset"]: r["offset"] + r["frames"]], dtype=np.float32)
        if train:
            rate = random.uniform(0.9, 1.1)  # tempo: nearest-frame resampling along time
            idx = np.clip((np.arange(int(len(m) / rate)) * rate).astype(int), 0, len(m) - 1)
            m = m[idx]
            if random.random() < 0.5:  # add a noise floor, in the power domain
                level = m.mean() + random.uniform(-4.0, 0.0)
                m = np.logaddexp(m, level + np.random.randn(*m.shape) * 0.5)
        m = fx.normalize(m)
        if train:  # SpecAugment
            for _ in range(2):
                w = random.randint(0, 15)
                s = random.randint(0, fx.N_MELS - w)
                m[:, s: s + w] = 0
            for _ in range(2):
                w = random.randint(0, min(40, len(m) // 8))
                s = random.randint(0, len(m) - w)
                m[s: s + w] = 0
        return m


def batches(rows, max_frames: int, shuffle: bool):
    order = sorted(rows, key=lambda r: r["frames"] + (random.random() * 40 if shuffle else 0))
    out, cur, longest = [], [], 0
    for r in order:
        longest = max(longest, r["frames"])
        if cur and longest * (len(cur) + 1) > max_frames:
            out.append(cur)
            cur, longest = [], r["frames"]
        cur.append(r)
    if cur:
        out.append(cur)
    if shuffle:
        random.shuffle(out)
    return out


def collate(data, batch, train):
    mels = [data.load(r, train) for r in batch]
    lens = torch.tensor([len(m) for m in mels])
    x = torch.zeros(len(mels), int(lens.max()), fx.N_MELS)
    for i, m in enumerate(mels):
        x[i, : len(m)] = torch.from_numpy(m)
    y = torch.tensor([t for r in batch for t in r["ids"]])
    ylen = torch.tensor([len(r["ids"]) for r in batch])
    return x, lens, y, ylen, sum(r["frames"] for r in batch)


class BatchSet(torch.utils.data.Dataset):
    def __init__(self, data, batch_list, train):
        self.data, self.batch_list, self.train = data, batch_list, train

    def __len__(self):
        return len(self.batch_list)

    def __getitem__(self, i):
        return collate(self.data, self.batch_list[i], self.train)


def seed_worker(worker_id):
    np.random.seed((torch.initial_seed() + worker_id) % 2**32)


def greedy(log_probs, lengths):
    out = []
    for seq, n in zip(log_probs.argmax(-1).cpu().tolist(), lengths.cpu().tolist()):
        ids, prev = [], 0
        for t in seq[:n]:
            if t != prev and t != 0:
                ids.append(t)
            prev = t
        out.append(ids)
    return out


@torch.no_grad()
def evaluate(model, data, rows, device, max_frames=30000):
    """Phone error rate: edits needed to fix the output, per reference phone."""
    model.eval()
    errors = total = 0
    for b in batches(rows, max_frames, False):
        x, lens, _, _, _ = collate(data, b, False)
        lp, out_lens = model(x.to(device), lens.to(device))
        for ids, r in zip(greedy(lp.float(), out_lens), b):
            errors += edit_distance(ids, r["ids"])
            total += len(r["ids"])
    return errors / max(total, 1)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--epochs", type=int, default=30)
    ap.add_argument("--dim", type=int, default=144)
    ap.add_argument("--blocks", type=int, default=5)
    ap.add_argument("--lr", type=float, default=1.5e-3)
    ap.add_argument("--max-frames", type=int, default=24000)
    ap.add_argument("--limit", type=int, default=0, help="use only this many training clips (for speed tests)")
    ap.add_argument("--max-seconds", type=float, default=0, help="stop cleanly after this long")
    ap.add_argument("--workers", type=int, default=-1, help="data-loading processes (default: 2 on NVIDIA, 0 elsewhere)")
    ap.add_argument("--ckpt-dir", default=str(ROOT / "checkpoints"))
    ap.add_argument("--name", default="model")
    ap.add_argument("--resume", action="store_true")
    args = ap.parse_args()

    random.seed(7)
    torch.manual_seed(7)
    device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
    use_amp = device == "cuda"  # half precision on NVIDIA GPUs; the loss stays in full precision
    workers = args.workers if args.workers >= 0 else (min(2, os.cpu_count() or 1) if device == "cuda" else 0)
    data = Data()
    train_rows = data.split("train")
    if args.limit:
        train_rows = random.sample(train_rows, args.limit)
    val_ur = data.split("val", "common_voice")
    val_en = data.split("val", "librispeech")
    hours = sum(r["frames"] for r in train_rows) / 360000
    print(f"train clips {len(train_rows)} ({hours:.1f} h), val urdu {len(val_ur)}, val english {len(val_en)}, device {device}", flush=True)

    model = SpeechModel(len(phones.VOCAB), dim=args.dim, blocks=args.blocks).to(device)
    print("parameters", sum(p.numel() for p in model.parameters()), flush=True)
    opt = torch.optim.AdamW(model.parameters(), lr=args.lr, weight_decay=1e-4)
    steps_per_epoch = len(batches(train_rows, args.max_frames, False))
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=args.lr, total_steps=args.epochs * steps_per_epoch + 50, pct_start=0.12)
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)

    ckpt_dir = Path(args.ckpt_dir)
    ckpt_dir.mkdir(parents=True, exist_ok=True)
    (ROOT / "logs").mkdir(exist_ok=True)
    last_path, best_path = ckpt_dir / f"{args.name}_last.pt", ckpt_dir / f"{args.name}_best.pt"
    start, best = 0, 9.0
    if args.resume and last_path.exists():
        c = torch.load(last_path, map_location=device)
        model.load_state_dict(c["model"])
        opt.load_state_dict(c["opt"])
        sched.load_state_dict(c["sched"])
        start, best = c["epoch"] + 1, c["best"]
        print(f"resumed at epoch {start}, best {best:.4f}", flush=True)

    log = open(ROOT / "logs" / f"{args.name}.jsonl", "a")
    began = time.time()
    for epoch in range(start, args.epochs):
        model.train()
        t0, loss_sum, n, audio = time.time(), 0.0, 0, 0
        loader = torch.utils.data.DataLoader(
            BatchSet(data, batches(train_rows, args.max_frames, True), True), batch_size=None, shuffle=False,
            num_workers=workers, worker_init_fn=seed_worker, persistent_workers=False, pin_memory=device == "cuda")
        for x, lens, y, ylen, frames in loader:
            x, lens = x.to(device, non_blocking=True), lens.to(device)
            with torch.autocast("cuda", dtype=torch.float16, enabled=use_amp):
                lp, out_lens = model(x, lens)
            lp = lp.float().transpose(0, 1)
            if device == "mps":  # CTC loss is not implemented on the Apple GPU
                lp, out_lens = lp.cpu(), out_lens.cpu()
            else:
                y, ylen = y.to(device), ylen.to(device)
            loss = F.ctc_loss(lp, y, out_lens, ylen, blank=0, zero_infinity=True, reduction="sum") / ylen.sum()
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            scaler.step(opt)
            scaler.update()
            # Shuffled batching can give a few more batches per epoch than the estimate,
            # so stop stepping the schedule at its end instead of raising an error.
            if sched.last_epoch < sched.total_steps - 1:
                sched.step()
            loss_sum += loss.item()
            n += 1
            audio += frames
            if n % 100 == 0:
                print(f"epoch {epoch} step {n}/{steps_per_epoch} loss {loss_sum/n:.3f} speed {audio/100/(time.time()-t0):.0f}x realtime", flush=True)
        per_ur = evaluate(model, data, val_ur, device)
        per_en = evaluate(model, data, val_en, device)
        score = (per_ur + per_en) / 2
        rec = dict(epoch=epoch, loss=round(loss_sum / max(n, 1), 4), per_ur=round(per_ur, 4), per_en=round(per_en, 4),
                   minutes=round((time.time() - t0) / 60, 1), lr=sched.get_last_lr()[0])
        print(rec, flush=True)
        log.write(json.dumps(rec) + "\n")
        log.flush()
        state = dict(model=model.state_dict(), config=model.config, epoch=epoch, best=min(best, score), opt=opt.state_dict(), sched=sched.state_dict(), vocab=phones.VOCAB)
        torch.save(state, str(last_path) + ".tmp")
        os.replace(str(last_path) + ".tmp", last_path)  # never leave a half-written checkpoint
        if score < best:
            best = score
            torch.save(dict(model=model.state_dict(), config=model.config, epoch=epoch, score=score, vocab=phones.VOCAB), best_path)
        if args.max_seconds and time.time() - began > args.max_seconds:
            print("time budget reached", flush=True)
            break


if __name__ == "__main__":
    main()

In [ ]:
#@title 5. Download speech data (about 3.6 GB)
!python src/download_data.py all

In [ ]:
#@title 6. Medicine command clips: use commands.zip from Drive if present, otherwise generate them
import os, zipfile
zip_path = DRIVE_DIR + '/commands.zip'
os.makedirs('data/raw/commands', exist_ok=True)
if os.path.exists(zip_path):
    with zipfile.ZipFile(zip_path) as z:
        for m in z.namelist():
            name = os.path.basename(m)
            if name and (name.endswith('.flac') or name == 'manifest.jsonl'):
                with open('data/raw/commands/' + name, 'wb') as f:
                    f.write(z.read(m))
    print('using the clips from', zip_path)
else:
    print('no commands.zip on Drive, generating clips with espeak-ng')
    !python src/generate_commands.py

In [ ]:
#@title 7. Build features and phonetic labels
!python src/prepare.py

In [ ]:
#@title 8. Train (re-run after a disconnect to continue)
!python src/train.py --epochs {EPOCHS} --dim {DIM} --blocks {BLOCKS} --name {RUN_NAME} --ckpt-dir "{CKPT_DIR}" --resume 2>&1 | tee -a logs/train.log

In [ ]:
#@title 9. Results
import json
for line in open(f'logs/{RUN_NAME}.jsonl'):
    r = json.loads(line)
    print(f"epoch {r['epoch']:2d}  urdu errors {r['per_ur']:.1%}  english errors {r['per_en']:.1%}  ({r['minutes']} min)")
print('\nBest model saved at', f'{CKPT_DIR}/{RUN_NAME}_best.pt')

In [ ]:
#@title 10. Download the best model to this computer (optional, it is also on Drive)
from google.colab import files
files.download(f'{CKPT_DIR}/{RUN_NAME}_best.pt')